#### Model with Gating at position G (at attention calculation step) :  As in Paper where stated best effective

In [ ]:
### Aplied gated atention at position G elementwise

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

In [ ]:
import random
import numpy as np
import torch

SEED = 12

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

set_seed(SEED)
torch.use_deterministic_algorithms(True)

In [ ]:
# Rope

def compute_rope_params(seq_len, head_dim, device=None):
    # x: (seq_len, dim)

    assert head_dim % 2 == 0, "head_dim must be even for RoPE"

    theta = 1.0 / (10000 ** (torch.arange(0, head_dim, 2).float() / head_dim))

    pos = torch.arange(seq_len).float()

    angles = pos[:, None] * theta[None, :]


    angles = angles[None, None, :, :]

    return torch.cos(angles), torch.sin(angles)



# similar to sebastian
def apply_rope(x, cos, sin, offset=0):

    batch_size, num_heads, seq_len, head_dim = x.shape   # (batch_size, num_heads, seq_len, head_dim)

    assert head_dim % 2 == 0, "Head dimension must be even"

    cos_sel = cos[...,offset : offset + seq_len, :].to(x.device, x.dtype)  #(1,1,seq_len,head_dim//2)
    sin_sel = sin[..., offset : offset + seq_len, :].to(x.device, x.dtype)

    x_even = x[..., 0::2]  # (b,n_heads,seq_len,head_dim//2)
    x_odd  = x[..., 1::2]


    x_rot = torch.empty_like(x)

    x_rot[..., 0::2] = x_even * cos_sel - x_odd * sin_sel
    x_rot[..., 1::2] = x_even * sin_sel + x_odd * cos_sel

    return x_rot.to(dtype=x.dtype)

In [ ]:
import torch
from torch import nn


class causal_multi_head_transformer(nn.Module):
    def __init__(self, din, dout, context_length, dropout, ff_dim, num_heads, qk_norm, pre_norm, post_norm, vocab_size, qkv_bias=False):
        super().__init__()
        self.num_heads = num_heads

        assert dout % num_heads == 0, f"d_out must be divisible by num_heads {dout // num_heads}"

        self.head_dim = dout // num_heads

        self.context_length = context_length



        self.wq = nn.Linear(din, dout, bias = qkv_bias)
        self.wk = nn.Linear(din, dout, bias = qkv_bias)
        self.wv = nn.Linear(din, dout, bias = qkv_bias)
        self.dropout = nn.Dropout(dropout)

        self.out_proj = nn.Linear(dout, dout)  # Linear layer to combine head outputs


        # Feedforward
        self.ff = nn.Sequential(nn.Linear(din, ff_dim),
                                            nn.ReLU(), nn.Linear(ff_dim, din),)
        self.norm1 = nn.RMSNorm(din)
        self.norm2 = nn.RMSNorm(din)

        self.apply_rope = apply_rope

        self.gate = nn.Linear(din, self.num_heads * self.head_dim)

        nn.init.zeros_(self.gate.weight)
        nn.init.constant_(self.gate.bias, 2.0)  # sigmoid(2) ≈ 0.88
        # or for even closer to identity
        # nn.init.constant_(self.gate.bias, 4.0)  # sigmoid(4) ≈ 0.98

        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length, dtype=torch.bool), diagonal=1)
        )

        self.qk_norm  = qk_norm
        self.pre_norm = pre_norm
        self.post_norm = post_norm


        if self.qk_norm:
            self.q_norm = nn.RMSNorm(self.head_dim)
            self.k_norm = nn.RMSNorm(self.head_dim)

        if self.pre_norm and self.post_norm:
          self.post_attn_norm = nn.RMSNorm(dout)
          self.post_ff_norm = nn.RMSNorm(dout)
          self.pre_ff_norm = nn.RMSNorm(dout)

    def forward(self, x, cos, sin, start_pos):

        b, num_tokens, d_in = x.shape

        assert num_tokens <= self.context_length

        #Pre-LN
        if self.pre_norm:
          x_norm = self.norm1(x)

        else:
          x_norm =x

        q = self.wq(x_norm)
        k = self.wk(x_norm)
        v = self.wv(x_norm)

        k = k.view(b, num_tokens, self.num_heads, self.head_dim)
        v = v.view(b, num_tokens, self.num_heads, self.head_dim)
        q = q.view(b, num_tokens, self.num_heads, self.head_dim)


        k = k.transpose(1,2)  # reshape to (b, num_heads, num_tokens, head_dim)
        v = v.transpose(1,2)
        q = q.transpose(1,2)

        #QK normalization (optional)
        if self.qk_norm:
            q = self.q_norm(q)
            k = self.k_norm(k)

        q_enc = self.apply_rope(q, cos, sin, start_pos)
        k_enc = self.apply_rope(k, cos, sin, start_pos)

        attn_scores = q_enc @ k_enc.transpose(2,3)    # (b, num_heads, num_tokens, head_dim) @ (b, num_heads, head_dim, num_tokens)

        attn_scores = attn_scores / (self.head_dim ** 0.5)

        mask = self.mask[:num_tokens, :num_tokens]
        masked = attn_scores.masked_fill(mask.bool(), float('-inf'))

        attn_weights = torch.softmax(masked, dim=-1)                             # shape: (b, num_heads, num_tokens, num_tokens)
        attn_weights = self.dropout(attn_weights)
        z = (attn_weights @ v)

        #gate applied elemnetwise

        gate_val = self.gate(x_norm)

        gate_val = gate_val.view(b, num_tokens, self.num_heads, self.head_dim)
        gate_val = gate_val.transpose(1,2)


        z = z * torch.sigmoid(gate_val)

       # Reshape and project
        z = z.transpose(1,2).contiguous().view(b, num_tokens, -1)  # Reshape to (b, num_tokens, num_heads * head_dim)
        attn_out = self.out_proj(z)

        # after attention output
        if self.pre_norm and self.post_norm:
          attn_out = self.post_attn_norm(attn_out)
          x = x + attn_out

          ff_out = self.ff(self.pre_ff_norm(x))
          ff_out = self.post_ff_norm(ff_out)
          x = x + ff_out

        elif self.pre_norm:                       # Pre-LN
          x= x + attn_out
          x = x + self.ff(self.norm2(x))

        elif self.post_norm:                    # Post-LN
          x = self.norm1(x + attn_out)
          x = self.norm2(x + self.ff(x))

        sink_val = attn_weights[:, :, :, 0].mean(dim=-1)

        return x, gate_val, sink_val

In [ ]:
class Gated_Transformer_LM(nn.Module):
  def __init__(self, din, dout, context_length, dropout, ff_dim, num_heads, vocab_size, qk_norm, pre_norm, post_norm, n_transformer):
    super().__init__()


    assert din == dout

    self.embedding = nn.Embedding(vocab_size, din)


    self.dstack = nn.ModuleList([causal_multi_head_transformer(din, dout, context_length, dropout, ff_dim, num_heads, qk_norm, pre_norm, post_norm, vocab_size)
    for _ in range(n_transformer)])

    cos, sin = compute_rope_params(context_length, din//num_heads)

    # register so they move with the model and are saved in state_dict
    self.register_buffer("rope_cos", cos)
    self.register_buffer("rope_sin", sin)

    self.cos, self.sin = cos, sin

    self.final_norm = nn.RMSNorm(din)
    self.out_head = nn.Linear(
        din, vocab_size, bias=False
    )


  def forward(self, inp, start_pos: int = 0):
      gate_vals = []
      attn_sink_val = []

      x = self.embedding(inp)

      for layer in self.dstack:
        x, gate_val, sink_info = layer(x, self.cos, self.sin, start_pos)

        gate_vals.append(gate_val)
        attn_sink_val.append(sink_info)

      x = self.final_norm(x)
      logits = self.out_head(x)


      return logits, gate_vals, attn_sink_val # Only return logits, as gate_val is not computed

### Model without Gating

In [ ]:
import torch
from torch import nn

class causal_multi_head_transformer_no_gate(nn.Module):
    def __init__(self, din, dout, context_length, dropout, ff_dim, num_heads, qk_norm, pre_norm, post_norm, vocab_size, qkv_bias=False):
        super().__init__()
        self.num_heads = num_heads

        assert dout % num_heads == 0, f"d_out must be divisible by num_heads {dout // num_heads}"

        self.head_dim = dout // num_heads

        self.context_length = context_length

        self.wq = nn.Linear(din, dout, bias = qkv_bias)
        self.wk = nn.Linear(din, dout, bias = qkv_bias)
        self.wv = nn.Linear(din, dout, bias = qkv_bias)
        self.dropout = nn.Dropout(dropout)

        self.out_proj = nn.Linear(dout, dout)  # Linear layer to combine head outputs


        # Feedforward
        self.ff = nn.Sequential( nn.Linear(din, ff_dim),
                                            nn.ReLU(), nn.Linear(ff_dim, din), )
        self.norm1 = nn.RMSNorm(din)
        self.norm2 = nn.RMSNorm(din)

        self.apply_rope = apply_rope

        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length, dtype=torch.bool), diagonal=1)
        )

        self.qk_norm  = qk_norm
        self.pre_norm = pre_norm
        self.post_norm = post_norm


        if self.qk_norm:
          self.q_norm = nn.RMSNorm(self.head_dim)
          self.k_norm = nn.RMSNorm(self.head_dim)

        if self.pre_norm and self.post_norm:
          self.post_attn_norm = nn.RMSNorm(dout)
          self.post_ff_norm = nn.RMSNorm(dout)
          self.pre_ff_norm = nn.RMSNorm(dout)


    def forward(self, x, cos, sin, start_pos):

        b, num_tokens, d_in = x.shape

        assert num_tokens <= self.context_length

        #Pre-LN

        if self.pre_norm:
          x_norm = self.norm1(x)

        else:
          x_norm =x

        q = self.wq(x_norm)
        k = self.wk(x_norm)
        v = self.wv(x_norm)

        k = k.view(b, num_tokens, self.num_heads, self.head_dim)
        v = v.view(b, num_tokens, self.num_heads, self.head_dim)
        q = q.view(b, num_tokens, self.num_heads, self.head_dim)


        k = k.transpose(1,2)  # reshape to (b, num_heads, num_tokens, head_dim)
        v = v.transpose(1,2)
        q = q.transpose(1,2)

        #QK normalization (optional)
        # Qk normalization L2 norm return unit vectors q/|q| = 1, k/|k| = 1 which scales the values in q.k = cos (theta) and range of values [-1, 1]
        # and no magnitude scale is taken in consideration in SDPA q.k/root(head_dim) which limits the range of values to [-1/root(head_dim), 1/root(head_dim)].
        # can add learnable paramereter Y which can change the scale [-1/root(head_dim) * y, 1/root(head_dim)] * y]
        # if self.qk_norm:
        #     q = q / (q.norm(dim=-1, keepdim=True) + 1e-6)
        #     k = k / (k.norm(dim=-1, keepdim=True) + 1e-6)

        # QK norm with RMSNorm (common with LLM models)

        if self.qk_norm:
            q = self.q_norm(q)
            k = self.k_norm(k)



        q_enc = self.apply_rope(q, cos, sin, start_pos)
        k_enc = self.apply_rope(k, cos, sin, start_pos)

        attn_scores = q_enc @ k_enc.transpose(2,3)
        attn_scores = attn_scores / (self.head_dim ** 0.5)

        mask = self.mask[:num_tokens, :num_tokens]
        masked = attn_scores.masked_fill(mask.bool(), float('-inf'))

        attn_weights = torch.softmax(masked, dim=-1)
        attn_weights = self.dropout(attn_weights)
        z = (attn_weights @ v)

        # Gating mechanism removed

       # Reshape and project
        z = z.transpose(1,2).contiguous().view(b, num_tokens, -1)  # Reshape to (b, num_tokens, num_heads, head_dim)
        attn_out = self.out_proj(z)

        # after attention output
        if self.pre_norm and self.post_norm:
          attn_out = self.post_attn_norm(attn_out)
          x = x + attn_out

          ff_out = self.ff(self.pre_ff_norm(x))
          ff_out = self.post_ff_norm(ff_out)
          x = x + ff_out


        elif self.pre_norm:                       # Pre-LN
          x= x + attn_out
          x = x + self.ff(self.norm2(x))

        elif self.post_norm:                    # Post-LN
          x = self.norm1(x + attn_out)
          x = self.norm2(x + self.ff(x))


        sink_val = attn_weights[:, :, :, 0].mean(dim=-1)

        return x,  sink_val # Only return x, as gate_val is not computed

In [ ]:
class Transformer_No_Gate(nn.Module):
  def __init__(self, din, dout, context_length, dropout, ff_dim, num_heads, vocab_size, qk_norm, pre_norm, post_norm, n_transformer):
    super().__init__()

    self.embedding = nn.Embedding(vocab_size, din)


    self.dstack = nn.ModuleList([causal_multi_head_transformer_no_gate(din, dout, context_length, dropout, ff_dim, num_heads, qk_norm, pre_norm, post_norm, vocab_size)
    for _ in range(n_transformer)])

    cos, sin = compute_rope_params(context_length, din//num_heads)

    # register so they move with the model and are saved in state_dict
    self.register_buffer("rope_cos", cos)
    self.register_buffer("rope_sin", sin)

    self.cos, self.sin = cos, sin

    self.final_norm = nn.RMSNorm(din)
    self.out_head = nn.Linear(
        din, vocab_size, bias=False
    )


  def forward(self, inp, start_pos: int = 0):
      attn_sink_val = []
      x = self.embedding(inp)

      for layer in self.dstack:
        x, sink_info = layer(x, self.cos, self.sin, start_pos)
        attn_sink_val.append(sink_info)

      x = self.final_norm(x)
      logits = self.out_head(x)


      return logits, attn_sink_val # Only return logits, as gate_val is not computed


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from datasets import load_dataset
from transformers import AutoTokenizer


# -----------------------------
# Load Dataset
# -----------------------------
# dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
dataset = load_dataset("wikitext", "wikitext-103-raw-v1")

texts = dataset["train"]["text"]
texts = [t for t in texts if len(t.strip()) > 0]


# -----------------------------
# Tokenizer (FAST)
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained("gpt2", use_fast=True)
tokenizer.pad_token = tokenizer.eos_token


# -----------------------------
# FAST Tokenization (BATCHED)
# -----------------------------
encodings = tokenizer(
    texts,
    padding=False,
    truncation=False
)

# Flatten tokens + add EOS between docs
all_tokens = [
    token
    for ids in encodings["input_ids"]
    for token in (ids + [tokenizer.eos_token_id])
]

tokens = torch.tensor(all_tokens, dtype=torch.long)


# -----------------------------
# Lazy Dataset (NO stacking)
# -----------------------------
class WikiTextDataset(Dataset):

    def __init__(self, tokens, seq_len=64, stride=None):
        self.tokens = tokens
        self.seq_len = seq_len
        self.stride = stride if stride is not None else seq_len

        self.num_samples = (len(tokens) - (seq_len + 1)) // self.stride

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        i = idx * self.stride
        chunk = self.tokens[i : i + self.seq_len + 1]

        x = chunk[:-1]
        y = chunk[1:]

        return x, y


# -----------------------------
# Create Dataset
# -----------------------------
seq_len = 128

train_dataset = WikiTextDataset(
    tokens,
    seq_len=seq_len,
    stride=128
)

max_samples = 300000
train_dataset = Subset(train_dataset, range(min(max_samples, len(train_dataset))))


# # -----------------------------
# # DataLoader 
# # -----------------------------
# dataloader = DataLoader(
#     train_dataset,
#     batch_size=16,
#     shuffle=True,
#     num_workers=2,      # parallel loading
#     pin_memory=True     # faster GPU transfer
# )


# # -----------------------------
# # Example Batch
# # -----------------------------
# for x, y in dataloader:
#     print("Input shape:", x.shape)
#     print("Target shape:", y.shape)
#     break

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1063 > 1024). Running this sequence through the model will result in indexing errors


In [ ]:
dataloader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,      # parallel loading
    pin_memory=True     # faster GPU transfer
)


len(dataloader)

18750

In [ ]:
vocab_size = tokenizer.vocab_size
print(vocab_size)

50257


In [ ]:
outputs = tokenizer("a gaoal is good",  return_tensors="pt", add_special_tokens=True)['input_ids']
print(outputs)
print(tokenizer.decode(outputs, skip_special_tokens=False))

tensor([[  64,  308, 5488,  282,  318,  922]])
['a gaoal is good']


### Training without Gating

In [ ]:
import math
import copy
from torch.utils.data import DataLoader, random_split

def model_training_no_gate(dataloader_dataset, qk_norm, pre_norm, post_norm, vocab_size, n_transformer, seed,
                          val_ratio=0.2):

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(device)

    set_seed(seed)

    gen = torch.Generator()
    gen.manual_seed(seed)

    # -------------------- Train/Validation Split --------------------
    val_size = int(len(dataloader_dataset) * val_ratio)
    train_size = len(dataloader_dataset) - val_size
    split_gen = torch.Generator().manual_seed(seed)
    train_dataset, val_dataset = random_split(
      dataloader_dataset, [train_size, val_size], generator=split_gen
    )

    train_dataloader = DataLoader(
      train_dataset, batch_size=16, shuffle=True,
      num_workers=0,
      pin_memory=True,
      generator=gen, worker_init_fn=seed_worker
    )

    print('train_dataloader size: ', len(train_dataloader))

    val_dataloader = DataLoader(
      val_dataset, batch_size=16, shuffle=False,
      num_workers=0, pin_memory=True
    )

    print('val_dataloader size: ', len(val_dataloader))
    # ----------------------------------------------------------------------

    model_no_gate = Transformer_No_Gate(din=256, dout=256, context_length=128,  dropout=0.1, ff_dim=1024, num_heads=8,
                                        vocab_size = vocab_size, qk_norm = qk_norm, pre_norm = pre_norm, post_norm = post_norm, n_transformer = n_transformer).to(device)

    optimizer_no_gate = torch.optim.Adam(model_no_gate.parameters(), lr=1e-3)
    loss_fn_no_gate = nn.CrossEntropyLoss()

    loss_history_no_gate, max_act_history_no_gate, grad_norm_history_no_gate = [], [], []

    epoch_loss_no_gate = []
    val_epoch_loss_no_gate = []

    # Additional tracking variables with _no_gate suffix
    train_ppl_history_no_gate = []
    val_ppl_history_no_gate = []
    best_val_ppl_no_gate = float("inf")
    best_model_state_no_gate = copy.deepcopy(model_no_gate.state_dict())
    epochs_without_improvement_no_gate = 0
    patience_no_gate = 1

    print("\n--- Training without Gating ---")
    for epoch in range(10):
        model_no_gate.train()
        total_loss_no_gate=0

        for step, (xb, yb) in enumerate(train_dataloader):
            xb, yb = xb.to(device), yb.to(device)
            optimizer_no_gate.zero_grad()
            out_no_gate, attn_sink_info = model_no_gate(xb)

            # Reshape for CrossEntropyLoss: (batch * seq, vocab)
            loss_no_gate = loss_fn_no_gate(out_no_gate.reshape(-1, vocab_size), yb.reshape(-1))
            loss_no_gate.backward()

            total_norm_no_gate = torch.nn.utils.clip_grad_norm_(model_no_gate.parameters(), max_norm=1.0)
            optimizer_no_gate.step()

            loss_history_no_gate.append(loss_no_gate.item())
            max_act_history_no_gate.append(out_no_gate.abs().max().item())
            grad_norm_history_no_gate.append(total_norm_no_gate.item())

            total_loss_no_gate += loss_no_gate.item()

            if step % 500 == 0:
              out_no_gate_mean = out_no_gate.mean().item()
              max_act = out_no_gate.abs().max().item()

              # Accessing sink info for the last layer: attn_sink_info[-1]
              # attn_sink_info is a list of tensors
              sink_val_tensor = attn_sink_info[-1]
              avg_sink_val = sink_val_tensor.float().mean().item()

              print(f"Epoch {epoch} Step {step} | Loss={loss_no_gate.item():.4f} | "
                    f"MaxAct={max_act:.4f} | MeanAct={out_no_gate_mean:.4f} | "
                    f"GradNorm={total_norm_no_gate:.4f} | Sink Val={avg_sink_val:.4f}")



        avg_train_loss_no_gate = total_loss_no_gate / len(train_dataloader)
        epoch_loss_no_gate.append(avg_train_loss_no_gate)

        train_ppl_no_gate = math.exp(min(avg_train_loss_no_gate, 20))
        train_ppl_history_no_gate.append(train_ppl_no_gate)

        # ===================== Validation =====================
        model_no_gate.eval()
        total_val_loss_no_gate = 0.0

        with torch.no_grad():
            for step, (xb, yb) in enumerate(val_dataloader):
                xb, yb = xb.to(device), yb.to(device)

                val_out_no_gate, attn_sink_info = model_no_gate(xb)
                val_loss_no_gate = loss_fn_no_gate(val_out_no_gate.reshape(-1, vocab_size), yb.reshape(-1))
                total_val_loss_no_gate += val_loss_no_gate.item()

        avg_val_loss_no_gate = total_val_loss_no_gate / len(val_dataloader)
        val_epoch_loss_no_gate.append(avg_val_loss_no_gate)

        val_ppl_no_gate = math.exp(min(avg_val_loss_no_gate, 20))
        val_ppl_history_no_gate.append(val_ppl_no_gate)

        print(
            f"\nEpoch {epoch} Summary | "
            f"Train Loss={avg_train_loss_no_gate:.4f} | Train PPL={train_ppl_no_gate:.2f} | "
            f"Val Loss={avg_val_loss_no_gate:.4f} | Val PPL={val_ppl_no_gate:.2f}\n"
        )

        # Early Stopping Logic
        if val_ppl_no_gate < best_val_ppl_no_gate:
            best_val_ppl_no_gate = val_ppl_no_gate
            best_model_state_no_gate = copy.deepcopy(model_no_gate.state_dict())
            epochs_without_improvement_no_gate = 0
        else:
            epochs_without_improvement_no_gate += 1
            if epochs_without_improvement_no_gate >= patience_no_gate:
                print(f"Early stopping triggered at epoch {epoch}")
                break

    # Restore Best Model
    model_no_gate.load_state_dict(best_model_state_no_gate)

    # Updated return dictionary with _no_gate suffix
    return {
        "model_no_gate": model_no_gate,
        "train_loss_no_gate": epoch_loss_no_gate,
        "val_loss_no_gate": val_epoch_loss_no_gate,
        "train_ppl_no_gate": train_ppl_history_no_gate,
        "val_ppl_no_gate": val_ppl_history_no_gate,
    }

### config 1: pre norm =True, post_norm = False, qk norm = False

In [ ]:
model_det_no_gate = model_training_no_gate(train_dataset, qk_norm = False, pre_norm = True, post_norm = False, vocab_size = vocab_size,
                                                                              n_transformer = 8, seed=SEED)

cuda
train_dataloader size:  15000
val_dataloader size:  3750

--- Training without Gating ---
Epoch 0 Step 0 | Loss=11.0174 | MaxAct=3.2652 | MeanAct=0.0006 | GradNorm=1.7499 | Sink Val=0.0423
Epoch 0 Step 500 | Loss=5.9639 | MaxAct=15.5792 | MeanAct=-3.8247 | GradNorm=0.5696 | Sink Val=0.0326
Epoch 0 Step 1000 | Loss=5.6303 | MaxAct=16.9313 | MeanAct=-3.8176 | GradNorm=0.5666 | Sink Val=0.0309
Epoch 0 Step 1500 | Loss=5.3347 | MaxAct=18.1947 | MeanAct=-3.9398 | GradNorm=0.5899 | Sink Val=0.0308
Epoch 0 Step 2000 | Loss=5.5404 | MaxAct=18.7770 | MeanAct=-3.8532 | GradNorm=0.6247 | Sink Val=0.0277
Epoch 0 Step 2500 | Loss=5.0714 | MaxAct=19.5068 | MeanAct=-3.9133 | GradNorm=0.6444 | Sink Val=0.0319
Epoch 0 Step 3000 | Loss=4.9643 | MaxAct=20.5793 | MeanAct=-4.0544 | GradNorm=0.6073 | Sink Val=0.0291
Epoch 0 Step 3500 | Loss=4.7268 | MaxAct=20.9892 | MeanAct=-3.9564 | GradNorm=0.5978 | Sink Val=0.0291
Epoch 0 Step 4000 | Loss=4.8615 | MaxAct=19.9404 | MeanAct=-4.0368 | GradNorm=0.5842 |

In [ ]:
print('#### Logs for Config 1 (No Gate) - SEED 12')
print('train_loss_no_gate: ', model_det_no_gate['train_loss_no_gate'])
print('val_loss_no_gate: ', model_det_no_gate['val_loss_no_gate'])
print('train_ppl_no_gate: ', model_det_no_gate['train_ppl_no_gate'])
print('val_ppl_no_gate: ', model_det_no_gate['val_ppl_no_gate'])
print('\nBest Training PPL: ', min(model_det_no_gate['train_ppl_no_gate']))
print('Best Validation PPL: ', min(model_det_no_gate['val_ppl_no_gate']))

#### Logs for Config 1 (No Gate) - SEED 12
train_loss_no_gate:  [4.719488196150462, 4.0771412472724915, 3.8837324659824373, 3.771570744498571, 3.694350135056178, 3.636253528849284, 3.590219136444728, 3.5520580766677856, 3.5198732053438824, 3.4919002442995706]
val_loss_no_gate:  [4.2422461548487345, 4.017440444628398, 3.9163849721272785, 3.8563350107828778, 3.821792586135864, 3.795255835723877, 3.777996182378133, 3.7636579288482666, 3.7504323752085367, 3.7399949403127035]
train_ppl_no_gate:  [112.11085921264636, 58.97662902869534, 48.60529453306802, 43.448257376548646, 40.2194268977066, 37.949393753882525, 36.24201700311087, 34.885039743686335, 33.78014505042305, 32.84830824783357]
val_ppl_no_gate:  [69.56392784151853, 55.558718064588945, 50.2185746961658, 47.29170975957749, 45.68603110770787, 44.48961698305338, 43.728330287245534, 43.10581595030145, 42.53947103695511, 42.09777716286972]

Best Training PPL:  32.84830824783357
Best Validation PPL:  42.09777716286972


In [ ]:
SEED2 = 42
model_det_no_gate2 = model_training_no_gate(train_dataset, qk_norm = False, pre_norm = True, post_norm = False, vocab_size = vocab_size,
                                                                              n_transformer = 8, seed=SEED2)

cuda
train_dataloader size:  15000
val_dataloader size:  3750

--- Training without Gating ---
Epoch 0 Step 0 | Loss=10.9763 | MaxAct=3.3128 | MeanAct=0.0004 | GradNorm=1.5225 | Sink Val=0.0423
Epoch 0 Step 500 | Loss=5.8139 | MaxAct=15.8957 | MeanAct=-3.9162 | GradNorm=0.5633 | Sink Val=0.0275
Epoch 0 Step 1000 | Loss=5.7704 | MaxAct=17.6832 | MeanAct=-3.7067 | GradNorm=0.6045 | Sink Val=0.0309
Epoch 0 Step 1500 | Loss=5.4018 | MaxAct=17.7353 | MeanAct=-3.9593 | GradNorm=0.6063 | Sink Val=0.0283
Epoch 0 Step 2000 | Loss=5.0827 | MaxAct=19.0875 | MeanAct=-3.8475 | GradNorm=0.5232 | Sink Val=0.0261
Epoch 0 Step 2500 | Loss=4.9994 | MaxAct=19.2015 | MeanAct=-3.8154 | GradNorm=0.5656 | Sink Val=0.0281
Epoch 0 Step 3000 | Loss=4.8086 | MaxAct=19.6626 | MeanAct=-3.8497 | GradNorm=0.5406 | Sink Val=0.0258
Epoch 0 Step 3500 | Loss=4.8754 | MaxAct=20.7291 | MeanAct=-3.7962 | GradNorm=0.5934 | Sink Val=0.0249
Epoch 0 Step 4000 | Loss=5.0045 | MaxAct=20.7887 | MeanAct=-4.1494 | GradNorm=0.5689 |

In [ ]:
print('#### Logs for Config 1 (No Gate) - SEED 42')
print('train_loss_no_gate: ', model_det_no_gate2['train_loss_no_gate'])
print('val_loss_no_gate: ', model_det_no_gate2['val_loss_no_gate'])
print('train_ppl_no_gate: ', model_det_no_gate2['train_ppl_no_gate'])
print('val_ppl_no_gate: ', model_det_no_gate2['val_ppl_no_gate'])
print('\nBest Training PPL: ', min(model_det_no_gate2['train_ppl_no_gate']))
print('Best Validation PPL: ', min(model_det_no_gate2['val_ppl_no_gate']))

#### Logs for Config 1 (No Gate) - SEED 42
train_loss_no_gate:  [4.7249910514036815, 4.079431148894628, 3.88250361568133, 3.768916593472163, 3.690642867708206, 3.631970434776942, 3.5854698383967083, 3.5469140715122225, 3.514392034403483, 3.4863731524626416]
val_loss_no_gate:  [4.24147838897705, 4.013528550593058, 3.911636125055949, 3.854599902470907, 3.8181282243092856, 3.790183254114787, 3.7700879991531373, 3.7563224818547565, 3.7444968528111775, 3.739497846031189]
train_ppl_no_gate:  [112.72948959886, 59.11183445165414, 48.54560258599083, 43.333092040341704, 40.070598772894066, 37.78720052283095, 36.070300950624556, 34.70605167185522, 33.59549680758727, 32.667253445635836]
val_ppl_no_gate:  [69.5105395293061, 55.34180279816899, 49.98065972281819, 47.209724667961986, 45.51892731067917, 44.26451118659577, 43.38388241278952, 42.790772427491554, 42.28772491530835, 42.07685579895358]

Best Training PPL:  32.667253445635836
Best Validation PPL:  42.07685579895358


In [ ]:
SEED3 = 100
model_det_no_gate3 = model_training_no_gate(train_dataset, qk_norm = False, pre_norm = True, post_norm = False, vocab_size = vocab_size,
                                                                              n_transformer = 8, seed=SEED3)

cuda
train_dataloader size:  15000
val_dataloader size:  3750

--- Training without Gating ---
Epoch 0 Step 0 | Loss=10.9961 | MaxAct=3.2035 | MeanAct=0.0002 | GradNorm=1.4642 | Sink Val=0.0433
Epoch 0 Step 500 | Loss=6.0030 | MaxAct=16.5102 | MeanAct=-3.5936 | GradNorm=0.5404 | Sink Val=0.0261
Epoch 0 Step 1000 | Loss=5.5665 | MaxAct=16.9960 | MeanAct=-3.8171 | GradNorm=0.5807 | Sink Val=0.0269
Epoch 0 Step 1500 | Loss=5.2690 | MaxAct=17.9270 | MeanAct=-3.8073 | GradNorm=0.6128 | Sink Val=0.0274
Epoch 0 Step 2000 | Loss=5.3305 | MaxAct=18.3083 | MeanAct=-3.8080 | GradNorm=0.6185 | Sink Val=0.0275
Epoch 0 Step 2500 | Loss=5.1236 | MaxAct=19.2275 | MeanAct=-3.9190 | GradNorm=0.5970 | Sink Val=0.0279
Epoch 0 Step 3000 | Loss=4.9831 | MaxAct=19.9490 | MeanAct=-3.9734 | GradNorm=0.6037 | Sink Val=0.0280
Epoch 0 Step 3500 | Loss=4.9074 | MaxAct=20.2486 | MeanAct=-3.9538 | GradNorm=0.5706 | Sink Val=0.0301
Epoch 0 Step 4000 | Loss=4.8735 | MaxAct=20.9724 | MeanAct=-4.0217 | GradNorm=0.5677 |

In [ ]:
print('#### Logs for Config 1 (No Gate) - SEED 100')
print('train_loss_no_gate: ', model_det_no_gate3['train_loss_no_gate'])
print('val_loss_no_gate: ', model_det_no_gate3['val_loss_no_gate'])
print('train_ppl_no_gate: ', model_det_no_gate3['train_ppl_no_gate'])
print('val_ppl_no_gate: ', model_det_no_gate3['val_ppl_no_gate'])
print('\nBest Training PPL: ', min(model_det_no_gate3['train_ppl_no_gate']))
print('Best Validation PPL: ', min(model_det_no_gate3['val_ppl_no_gate']))

#### Logs for Config 1 (No Gate) - SEED 100
train_loss_no_gate:  [4.72543500494957, 4.07998109644254, 3.882702228562037, 3.7688320770104724, 3.6902694636821747, 3.6312562475363412, 3.584351127751668, 3.545844503434499, 3.5131828091144564, 3.4848310912450153]
val_loss_no_gate:  [4.245217866452535, 4.020862634150187, 3.9200995871861775, 3.865856542015076, 3.8246924271265668, 3.796362634531657, 3.7801012776056924, 3.7662271405537924, 3.7539064977645875, 3.744760188293457]
train_ppl_no_gate:  [112.77954736633309, 59.14435180066206, 48.55524532552052, 43.329429835488824, 40.05563904317225, 37.76022302100001, 36.02997128380118, 34.66895103125047, 33.554896835453825, 32.616917361647175]
val_ppl_no_gate:  [69.770959239536, 55.74917623349127, 50.405464266477246, 47.74414979387339, 45.81870560792108, 44.53888529811551, 43.82047954409338, 43.21670630662041, 42.687515386011164, 42.29886224010548]

Best Training PPL:  32.616917361647175
Best Validation PPL:  42.29886224010548


### config 2: pre norm = True, post_norm = False, qk norm = True

In [ ]:
model_det_no_gate = model_training_no_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = False, vocab_size = vocab_size,
                                                                              n_transformer = 8, seed=SEED)

cuda
train_dataloader size:  15000
val_dataloader size:  3750

--- Training without Gating ---
Epoch 0 Step 0 | Loss=11.0154 | MaxAct=3.2417 | MeanAct=0.0006 | GradNorm=1.7399 | Sink Val=0.0424
Epoch 0 Step 500 | Loss=5.9425 | MaxAct=15.8473 | MeanAct=-3.7970 | GradNorm=0.5702 | Sink Val=0.0307
Epoch 0 Step 1000 | Loss=5.5844 | MaxAct=17.2085 | MeanAct=-3.8148 | GradNorm=0.5575 | Sink Val=0.0336
Epoch 0 Step 1500 | Loss=5.3255 | MaxAct=18.1294 | MeanAct=-3.9266 | GradNorm=0.6195 | Sink Val=0.0321
Epoch 0 Step 2000 | Loss=5.5031 | MaxAct=18.9850 | MeanAct=-3.8773 | GradNorm=0.6118 | Sink Val=0.0333
Epoch 0 Step 2500 | Loss=5.0744 | MaxAct=19.3436 | MeanAct=-3.9459 | GradNorm=0.6240 | Sink Val=0.0328
Epoch 0 Step 3000 | Loss=4.9387 | MaxAct=19.8866 | MeanAct=-4.0712 | GradNorm=0.5835 | Sink Val=0.0313
Epoch 0 Step 3500 | Loss=4.7267 | MaxAct=20.6365 | MeanAct=-3.9599 | GradNorm=0.5826 | Sink Val=0.0331
Epoch 0 Step 4000 | Loss=4.8449 | MaxAct=20.0442 | MeanAct=-4.0510 | GradNorm=0.5788 |

In [ ]:
print('#### Logs for Config 2 (No Gate) - SEED 12')
print('train_loss_no_gate: ', model_det_no_gate['train_loss_no_gate'])
print('val_loss_no_gate: ', model_det_no_gate['val_loss_no_gate'])
print('train_ppl_no_gate: ', model_det_no_gate['train_ppl_no_gate'])
print('val_ppl_no_gate: ', model_det_no_gate['val_ppl_no_gate'])
print('\nBest Training PPL: ', min(model_det_no_gate['train_ppl_no_gate']))
print('Best Validation PPL: ', min(model_det_no_gate['val_ppl_no_gate']))

#### Logs for Config 2 (No Gate) - SEED 12
train_loss_no_gate:  [4.712412454271316, 4.066887979141871, 3.8727493121147156, 3.7605480034510292, 3.6832162594000497, 3.6249245700041453, 3.5788403319835664, 3.54066681543986, 3.5083493403116863, 3.480303757127126]
val_loss_no_gate:  [4.233359502919515, 4.0061597819646195, 3.9036323252360026, 3.8467472931543987, 3.8097984102884928, 3.7841302474975587, 3.7651725905100504, 3.7544829750696818, 3.740701077906291, 3.7284124894460042]
train_ppl_no_gate:  [111.32039158143758, 58.375015361547284, 48.07437602131129, 42.971968314467105, 39.77411243778626, 37.52189277663785, 35.831963561689754, 34.48990992917601, 33.39310161947779, 32.469583445443696]
val_ppl_no_gate:  [68.94847613083991, 54.93550066953137, 49.58222116577596, 46.840456899489865, 45.14133791973265, 43.997387075075984, 43.17115614877612, 42.712130864292654, 42.127514484015414, 41.61299463178775]

Best Training PPL:  32.469583445443696
Best Validation PPL:  41.61299463178775


In [ ]:
SEED2 = 42
model_det_no_gate2 = model_training_no_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = False, vocab_size = vocab_size,
                                                                              n_transformer = 8, seed=SEED2)

cuda
train_dataloader size:  15000
val_dataloader size:  3750

--- Training without Gating ---
Epoch 0 Step 0 | Loss=10.9736 | MaxAct=3.2685 | MeanAct=0.0004 | GradNorm=1.5103 | Sink Val=0.0424
Epoch 0 Step 500 | Loss=5.7853 | MaxAct=16.0347 | MeanAct=-3.9252 | GradNorm=0.5597 | Sink Val=0.0364
Epoch 0 Step 1000 | Loss=5.7253 | MaxAct=17.4054 | MeanAct=-3.6800 | GradNorm=0.6101 | Sink Val=0.0356
Epoch 0 Step 1500 | Loss=5.3823 | MaxAct=17.8213 | MeanAct=-4.0079 | GradNorm=0.5649 | Sink Val=0.0334
Epoch 0 Step 2000 | Loss=5.0703 | MaxAct=18.8828 | MeanAct=-3.8901 | GradNorm=0.5238 | Sink Val=0.0321
Epoch 0 Step 2500 | Loss=5.0053 | MaxAct=19.3332 | MeanAct=-3.8412 | GradNorm=0.5660 | Sink Val=0.0333
Epoch 0 Step 3000 | Loss=4.7956 | MaxAct=20.3816 | MeanAct=-3.9343 | GradNorm=0.5460 | Sink Val=0.0323
Epoch 0 Step 3500 | Loss=4.8926 | MaxAct=20.9306 | MeanAct=-3.8187 | GradNorm=0.6136 | Sink Val=0.0299
Epoch 0 Step 4000 | Loss=5.0080 | MaxAct=21.2397 | MeanAct=-4.1659 | GradNorm=0.5916 |

In [ ]:
print('#### Logs for Config 2 (No Gate) - SEED 42')
print('train_loss_no_gate: ', model_det_no_gate2['train_loss_no_gate'])
print('val_loss_no_gate: ', model_det_no_gate2['val_loss_no_gate'])
print('train_ppl_no_gate: ', model_det_no_gate2['train_ppl_no_gate'])
print('val_ppl_no_gate: ', model_det_no_gate2['val_ppl_no_gate'])
print('\nBest Training PPL: ', min(model_det_no_gate2['train_ppl_no_gate']))
print('Best Validation PPL: ', min(model_det_no_gate2['val_ppl_no_gate']))

#### Logs for Config 2 (No Gate) - SEED 42
train_loss_no_gate:  [4.715883531665802, 4.065994276428222, 3.8679970618247985, 3.754073392009735, 3.6759613392035164, 3.6173044018745424, 3.5708078627586364, 3.532274544207255, 3.4995263869126636, 3.471427609014511]
val_loss_no_gate:  [4.230673728370666, 3.9993479221343993, 3.898301279258728, 3.8415824109395347, 3.8045747069676716, 3.777033613204956, 3.7580130526860556, 3.744377724838257, 3.7319516600290936, 3.7272425830841063]
train_ppl_no_gate:  [111.70746466788789, 58.322868757161125, 47.846456548059905, 42.694640281204194, 39.48659863209918, 37.23705627445485, 35.54529727748534, 34.20167242563397, 33.099771760718205, 32.18265391197899]
val_ppl_no_gate:  [68.76354452193526, 54.56255939320025, 49.31859937936291, 46.599155140708504, 44.90614777949735, 43.686258996028926, 42.86317444232951, 42.28268756441044, 41.76053104252519, 41.5643397909837]

Best Training PPL:  32.18265391197899
Best Validation PPL:  41.5643397909837


In [ ]:
SEED3 = 100
model_det_no_gate3 = model_training_no_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = False, vocab_size = vocab_size,
                                                                              n_transformer = 8, seed=SEED3)

cuda
train_dataloader size:  15000
val_dataloader size:  3750

--- Training without Gating ---
Epoch 0 Step 0 | Loss=10.9971 | MaxAct=3.1854 | MeanAct=0.0002 | GradNorm=1.4560 | Sink Val=0.0435
Epoch 0 Step 500 | Loss=5.9882 | MaxAct=15.9885 | MeanAct=-3.5996 | GradNorm=0.5464 | Sink Val=0.0284
Epoch 0 Step 1000 | Loss=5.5365 | MaxAct=17.1709 | MeanAct=-3.8057 | GradNorm=0.6056 | Sink Val=0.0287
Epoch 0 Step 1500 | Loss=5.2720 | MaxAct=17.7495 | MeanAct=-3.8218 | GradNorm=0.5946 | Sink Val=0.0303
Epoch 0 Step 2000 | Loss=5.2974 | MaxAct=18.9000 | MeanAct=-3.8165 | GradNorm=0.6170 | Sink Val=0.0292
Epoch 0 Step 2500 | Loss=5.0932 | MaxAct=18.8869 | MeanAct=-3.8919 | GradNorm=0.5665 | Sink Val=0.0270
Epoch 0 Step 3000 | Loss=4.9519 | MaxAct=19.5092 | MeanAct=-3.9880 | GradNorm=0.5890 | Sink Val=0.0267
Epoch 0 Step 3500 | Loss=4.9240 | MaxAct=19.9793 | MeanAct=-3.9940 | GradNorm=0.5462 | Sink Val=0.0318
Epoch 0 Step 4000 | Loss=4.8640 | MaxAct=20.1890 | MeanAct=-3.9622 | GradNorm=0.5654 |

In [ ]:
print('#### Logs for Config 2 (No Gate) - SEED 100')
print('train_loss_no_gate: ', model_det_no_gate3['train_loss_no_gate'])
print('val_loss_no_gate: ', model_det_no_gate3['val_loss_no_gate'])
print('train_ppl_no_gate: ', model_det_no_gate3['train_ppl_no_gate'])
print('val_ppl_no_gate: ', model_det_no_gate3['val_ppl_no_gate'])
print('\nBest Training PPL: ', min(model_det_no_gate3['train_ppl_no_gate']))
print('Best Validation PPL: ', min(model_det_no_gate3['val_ppl_no_gate']))

#### Logs for Config 2 (No Gate) - SEED 100
train_loss_no_gate:  [4.713237859328588, 4.063344417842229, 3.8682756171544392, 3.7557413735230765, 3.6783192612012225, 3.620175657018026, 3.5737378135363262, 3.535552968454361, 3.503243775590261, 3.475183166138331]
val_loss_no_gate:  [4.22716715888977, 4.006945736630757, 3.905257891146342, 3.8506311401367186, 3.810398654047648, 3.7831852473576864, 3.76690751024882, 3.752817618560791, 3.742452823829651, 3.734448768679301]
train_ppl_no_gate:  [111.412313926992, 58.16852598619664, 47.859786289984925, 42.76591357665713, 39.579814806674115, 37.34412700394392, 35.64959596928967, 34.31398401973808, 33.22304546366665, 32.30374494696668]
val_ppl_no_gate:  [68.52284264089117, 54.97869445457417, 49.66288587940347, 47.02273180145987, 45.168441859770546, 43.9558291773383, 43.24611964874902, 42.64105913545824, 42.20137586006861, 41.86494193424422]

Best Training PPL:  32.30374494696668
Best Validation PPL:  41.86494193424422


### config 3: pre_norm = True, post norm = True, qk norm = True

In [ ]:
model_det_no_gate = model_training_no_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = True, vocab_size = vocab_size,
                                                                              n_transformer = 8, seed=SEED)

cuda
train_dataloader size:  15000
val_dataloader size:  3750

--- Training without Gating ---
Epoch 0 Step 0 | Loss=11.0188 | MaxAct=3.2044 | MeanAct=-0.0000 | GradNorm=3.1944 | Sink Val=0.0445
Epoch 0 Step 500 | Loss=6.0062 | MaxAct=15.4789 | MeanAct=-3.7519 | GradNorm=0.5892 | Sink Val=0.0367
Epoch 0 Step 1000 | Loss=5.6218 | MaxAct=17.2794 | MeanAct=-3.8463 | GradNorm=0.5826 | Sink Val=0.0332
Epoch 0 Step 1500 | Loss=5.3338 | MaxAct=18.2676 | MeanAct=-3.9184 | GradNorm=0.6271 | Sink Val=0.0336
Epoch 0 Step 2000 | Loss=5.5248 | MaxAct=18.9323 | MeanAct=-3.8643 | GradNorm=0.6264 | Sink Val=0.0319
Epoch 0 Step 2500 | Loss=5.0269 | MaxAct=19.2536 | MeanAct=-3.8953 | GradNorm=0.5981 | Sink Val=0.0332
Epoch 0 Step 3000 | Loss=4.9775 | MaxAct=20.5794 | MeanAct=-3.9992 | GradNorm=0.6094 | Sink Val=0.0313
Epoch 0 Step 3500 | Loss=4.7461 | MaxAct=20.1889 | MeanAct=-4.0405 | GradNorm=0.5937 | Sink Val=0.0324
Epoch 0 Step 4000 | Loss=4.8475 | MaxAct=20.6911 | MeanAct=-4.0336 | GradNorm=0.5790 

In [ ]:
print('#### Logs for Config 3 (No Gate) - SEED 12')
print('train_loss_no_gate: ', model_det_no_gate['train_loss_no_gate'])
print('val_loss_no_gate: ', model_det_no_gate['val_loss_no_gate'])
print('train_ppl_no_gate: ', model_det_no_gate['train_ppl_no_gate'])
print('val_ppl_no_gate: ', model_det_no_gate['val_ppl_no_gate'])
print('\nBest Training PPL: ', min(model_det_no_gate['train_ppl_no_gate']))
print('Best Validation PPL: ', min(model_det_no_gate['val_ppl_no_gate']))

#### Logs for Config 3 (No Gate) - SEED 12
train_loss_no_gate:  [4.72731697529157, 4.071902536694209, 3.8705114835898082, 3.752848777929942, 3.6707820029735565, 3.6080569364706676, 3.5579122698783876, 3.5156624771118166, 3.4795779829661053, 3.4476841219266254]
val_loss_no_gate:  [4.246080546442668, 4.013217852783203, 3.907682739384969, 3.8446110368092854, 3.808924256960551, 3.7790907812754315, 3.75855570081075, 3.7451468229929605, 3.7347536520004274, 3.720465465418498]
train_ppl_no_gate:  [112.9919949770079, 58.66847540689966, 47.966914096859945, 42.64238782466129, 39.282612972366636, 36.89429516318929, 35.089862457409296, 33.63820508500476, 32.44602641033763, 31.427525648614736]
val_ppl_no_gate:  [69.8311752197905, 55.32461089212825, 49.7834569649668, 46.74050048051082, 45.101894711180066, 43.77622147539781, 42.886440375595704, 42.315219609956635, 41.8777078027312, 41.28360572790896]

Best Training PPL:  31.427525648614736
Best Validation PPL:  41.28360572790896


In [ ]:
SEED2 = 42
model_det_no_gate2 = model_training_no_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = True, vocab_size = vocab_size,
                                                                              n_transformer = 8, seed=SEED2)

cuda
train_dataloader size:  15000
val_dataloader size:  3750

--- Training without Gating ---
Epoch 0 Step 0 | Loss=10.9820 | MaxAct=3.1456 | MeanAct=0.0004 | GradNorm=2.9834 | Sink Val=0.0453
Epoch 0 Step 500 | Loss=5.8583 | MaxAct=15.8812 | MeanAct=-3.8371 | GradNorm=0.6003 | Sink Val=0.0381
Epoch 0 Step 1000 | Loss=5.7706 | MaxAct=17.2032 | MeanAct=-3.6328 | GradNorm=0.6600 | Sink Val=0.0358
Epoch 0 Step 1500 | Loss=5.3962 | MaxAct=17.9128 | MeanAct=-3.9217 | GradNorm=0.6221 | Sink Val=0.0332
Epoch 0 Step 2000 | Loss=5.0754 | MaxAct=18.7268 | MeanAct=-3.7838 | GradNorm=0.5532 | Sink Val=0.0317
Epoch 0 Step 2500 | Loss=5.0143 | MaxAct=19.0129 | MeanAct=-3.7901 | GradNorm=0.5874 | Sink Val=0.0326
Epoch 0 Step 3000 | Loss=4.7983 | MaxAct=20.7519 | MeanAct=-3.8139 | GradNorm=0.5602 | Sink Val=0.0320
Epoch 0 Step 3500 | Loss=4.8727 | MaxAct=20.9655 | MeanAct=-3.8046 | GradNorm=0.5488 | Sink Val=0.0302
Epoch 0 Step 4000 | Loss=4.9766 | MaxAct=20.8502 | MeanAct=-4.1026 | GradNorm=0.5684 |

In [ ]:
print('#### Logs for Config 3 (No Gate) - SEED 42')
print('train_loss_no_gate: ', model_det_no_gate2['train_loss_no_gate'])
print('val_loss_no_gate: ', model_det_no_gate2['val_loss_no_gate'])
print('train_ppl_no_gate: ', model_det_no_gate2['train_ppl_no_gate'])
print('val_ppl_no_gate: ', model_det_no_gate2['val_ppl_no_gate'])
print('\nBest Training PPL: ', min(model_det_no_gate2['train_ppl_no_gate']))
print('Best Validation PPL: ', min(model_det_no_gate2['val_ppl_no_gate']))

#### Logs for Config 3 (No Gate) - SEED 42
train_loss_no_gate:  [4.726772703313827, 4.072268147579829, 3.868449057976405, 3.74975187125206, 3.6667372694333396, 3.603740099096298, 3.5530822545528413, 3.5104637832164762, 3.4739425309181216, 3.441907184012731]
val_loss_no_gate:  [4.242350159263611, 4.007428758557637, 3.902249053764343, 3.8415842989603677, 3.803310805130005, 3.7741157035191852, 3.7540665878931683, 3.740051978556315, 3.7265374117533367, 3.722154767290751]
train_ppl_no_gate:  [112.93051333331827, 58.68992916177532, 47.8680878505521, 42.510532606083146, 39.1240461665465, 36.73537176096485, 34.92078653232161, 33.46378412699493, 32.26369263353346, 31.24649419136333]
val_ppl_no_gate:  [69.57116317337693, 55.00525678408431, 49.51368290817272, 46.59924312096726, 44.849426669304776, 43.558972233719665, 42.69434978288324, 42.10017841461997, 41.53504014203111, 41.353405139898044]

Best Training PPL:  31.24649419136333
Best Validation PPL:  41.353405139898044


In [ ]:
SEED3 = 100
model_det_no_gate3 = model_training_no_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = True, vocab_size = vocab_size,
                                                                              n_transformer = 8, seed=SEED3)

cuda
train_dataloader size:  15000
val_dataloader size:  3750

--- Training without Gating ---
Epoch 0 Step 0 | Loss=10.9733 | MaxAct=3.1086 | MeanAct=-0.0007 | GradNorm=3.1199 | Sink Val=0.0442
Epoch 0 Step 500 | Loss=6.0238 | MaxAct=15.6128 | MeanAct=-3.5437 | GradNorm=0.5951 | Sink Val=0.0306
Epoch 0 Step 1000 | Loss=5.5559 | MaxAct=16.8202 | MeanAct=-3.7131 | GradNorm=0.6328 | Sink Val=0.0286
Epoch 0 Step 1500 | Loss=5.2823 | MaxAct=17.8640 | MeanAct=-3.7579 | GradNorm=0.6484 | Sink Val=0.0248
Epoch 0 Step 2000 | Loss=5.3474 | MaxAct=18.7418 | MeanAct=-3.6614 | GradNorm=0.6226 | Sink Val=0.0306
Epoch 0 Step 2500 | Loss=5.0695 | MaxAct=19.1898 | MeanAct=-3.8309 | GradNorm=0.5976 | Sink Val=0.0259
Epoch 0 Step 3000 | Loss=4.9809 | MaxAct=20.4670 | MeanAct=-3.9106 | GradNorm=0.6090 | Sink Val=0.0243
Epoch 0 Step 3500 | Loss=4.9050 | MaxAct=20.7941 | MeanAct=-3.8482 | GradNorm=0.5602 | Sink Val=0.0252
Epoch 0 Step 4000 | Loss=4.8695 | MaxAct=20.7288 | MeanAct=-3.8492 | GradNorm=0.5611 

In [ ]:
print('#### Logs for Config 3 (No Gate) - SEED 100')
print('train_loss_no_gate: ', model_det_no_gate3['train_loss_no_gate'])
print('val_loss_no_gate: ', model_det_no_gate3['val_loss_no_gate'])
print('train_ppl_no_gate: ', model_det_no_gate3['train_ppl_no_gate'])
print('val_ppl_no_gate: ', model_det_no_gate3['val_ppl_no_gate'])
print('\nBest Training PPL: ', min(model_det_no_gate3['train_ppl_no_gate']))
print('Best Validation PPL: ', min(model_det_no_gate3['val_ppl_no_gate']))

#### Logs for Config 3 (No Gate) - SEED 100
train_loss_no_gate:  [4.724824139038722, 4.0688354258060455, 3.8656405188878376, 3.747217764377594, 3.6641601167996725, 3.60110495373408, 3.5504528685410817, 3.507837468067805, 3.4715021254857383, 3.43964028523763]
val_loss_no_gate:  [4.2441946285883585, 4.012006750996908, 3.9077038368225097, 3.84720690313975, 3.8086902132670084, 3.775901742998759, 3.7590136209487914, 3.74673078250885, 3.7330318534851075, 3.7238307919184366]
train_ppl_no_gate:  [112.71067522336999, 58.488808357415586, 47.73383706726141, 42.40294275283801, 39.023347341783804, 36.63869614952703, 34.82908691437198, 33.376012991591935, 32.18505213913584, 31.175741776534707]
val_ppl_no_gate:  [69.69960346541303, 55.25764771484745, 49.78450727942011, 46.86199018927773, 45.09134013232059, 43.63683979438813, 42.90608343742614, 42.38229831567727, 41.80566486711939, 41.42277257988196]

Best Training PPL:  31.175741776534707
Best Validation PPL:  41.42277257988196


### Training with Gating

In [ ]:
import math
import copy
from torch.utils.data import DataLoader, random_split

def model_training_with_gate(dataloader_dataset, qk_norm, pre_norm, post_norm,
                             vocab_size, n_transformer, seed,
                             val_ratio=0.2, patience=1):

    device = "cuda" if torch.cuda.is_available() else "cpu"

    set_seed(seed)

    gen = torch.Generator()
    gen.manual_seed(seed)

    # -------------------- Train/Validation Split --------------------
    val_size = int(len(dataloader_dataset) * val_ratio)
    train_size = len(dataloader_dataset) - val_size
    split_gen = torch.Generator().manual_seed(seed)
    train_dataset, val_dataset = random_split(
      dataloader_dataset, [train_size, val_size], generator=split_gen
    )

    train_dataloader = DataLoader(
      train_dataset, batch_size=16, shuffle=True,
      num_workers=0,
      pin_memory=True,
      generator=gen, worker_init_fn=seed_worker
    )

    print('train_dataloader size: ', len(train_dataloader))

    val_dataloader = DataLoader(
      val_dataset, batch_size=16, shuffle=False,
      num_workers=0, pin_memory=True
    )

    print('val_dataloader size: ', len(val_dataloader))

    # ----------------------------------------------------------------------

    model = Gated_Transformer_LM(
      din=256, dout=256, context_length=128, dropout=0.1,
      ff_dim=1024, num_heads=8,
      vocab_size=vocab_size, qk_norm=qk_norm,
      pre_norm=pre_norm, post_norm=post_norm,
      n_transformer=n_transformer
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    loss_history, gate_mean_history, max_act_history, grad_norm_history = [], [], [], []

    print("\n--- Training with Gating ---")
    gated_epoch_loss = []


    train_ppl_history = []
    val_ppl_history = []
    val_epoch_loss = []

    best_val_ppl = float("inf")
    best_model_state = copy.deepcopy(model.state_dict())
    epochs_without_improvement = 0

    for epoch in range(10):
        model.train()
        total_loss = 0

        for step, (xb, yb) in enumerate(train_dataloader):
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out, gate_vals, attn_sink_info = model(xb)
            loss = loss_fn(out.view(-1, vocab_size), yb.view(-1))
            loss.backward()

            total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            # Existing statistics
            gates = [torch.sigmoid(g) for g in gate_vals]
            layer_means = [g.mean().item() for g in gates]
            layer_std = [g.std().item() for g in gates]
            layer_sparsity = [(g < 0.1).float().mean().item() for g in gates]

            all_gates_combined = torch.stack(gate_vals)
            g = torch.sigmoid(all_gates_combined)

            gate_mean = g.mean().item()
            sparsity = (g < 0.1).float().mean()
            head_mean = g.mean(dim=(0, 1, 3, 4))

            dead = torch.mean(torch.stack([
              (g < 0.01).float().mean() for g in gates
            ])).item()

            open_ = torch.mean(torch.stack([
              (g > 0.99).float().mean() for g in gates
            ])).item()

            loss_history.append(loss.item())
            gate_mean_history.append(gate_mean)
            max_act_history.append(out.abs().max().item())
            grad_norm_history.append(total_norm.item())
            total_loss += loss.item()

            if step % 500 == 0:
              out_mean = out.mean().item()
              sink_val_tensor = attn_sink_info[-1]
              avg_sink_val = sink_val_tensor.float().mean().item()

              print(
                  f"Epoch {epoch} Step {step} | Loss={loss.item():.4f} | "
                  f"gate_mean={gate_mean:.4f} | "
                  f"LayerMeans={[round(m, 3) for m in layer_means]} | "
                  f"LayerStd={[round(s, 3) for s in layer_std]} | "
                  f"LayerSparsity={[round(s, 3) for s in layer_sparsity]} | "
                  f"MaxAct={out.abs().max():.4f} | MeanAct={out_mean:.4f} | GradNorm={total_norm:.4f} | "
                  f"Sparsity={sparsity.item():.4f} | HeadMeanAvg={head_mean.mean().item():.4f} | "
                  f"Sink Val={avg_sink_val:.4f} | Dead={dead:.3f} | Open={open_:.3f}"
              )

        # Training PPL
        avg_train_loss = total_loss / len(train_dataloader)
        train_ppl = math.exp(min(avg_train_loss, 20))
        train_ppl_history.append(train_ppl)


        gated_epoch_loss.append(avg_train_loss)

        # Validation

        model.eval()
        total_val_loss = 0.0

        with torch.no_grad():
            for xb, yb in val_dataloader:
                xb, yb = xb.to(device), yb.to(device)
                val_out, _, _ = model(xb)
                val_loss = loss_fn(val_out.view(-1, vocab_size), yb.view(-1))
                total_val_loss += val_loss.item()

        avg_val_loss = total_val_loss / len(val_dataloader)
        val_ppl = math.exp(min(avg_val_loss, 20))

        val_epoch_loss.append(avg_val_loss)
        val_ppl_history.append(val_ppl)

        print(
            f"\nEpoch {epoch} Summary | "
            f"Train Loss={avg_train_loss:.4f} | Train PPL={train_ppl:.2f} | "
            f"Val Loss={avg_val_loss:.4f} | Val PPL={val_ppl:.2f}\n"
        )

        # Early Stopping
        if val_ppl < best_val_ppl:
            best_val_ppl = val_ppl
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping triggered at epoch {epoch}.")
                break

    # -------------------- Load Best Model ------------------
    model.load_state_dict(best_model_state)
    # --------------------------------------------------------------

    return {
    "model": model,
    "train_loss": gated_epoch_loss,
    "val_loss": val_epoch_loss,
    "train_ppl": train_ppl_history,
    "val_ppl": val_ppl_history,
    "step_loss": loss_history,
    "gate_mean_history": gate_mean_history,
    "max_activation": max_act_history,
    "grad_norm": grad_norm_history,
    }

### config 1: pre norm =True, post_norm = False, qk norm = False

In [ ]:
model_det = model_training_with_gate(train_dataset, qk_norm = False, pre_norm = True, post_norm = False, vocab_size = vocab_size,
                                                                               n_transformer = 8, seed=SEED)

train_dataloader size:  15000
val_dataloader size:  3750

--- Training with Gating ---
Epoch 0 Step 0 | Loss=10.9586 | gate_mean=0.8808 | LayerMeans=[0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881] | LayerStd=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | LayerSparsity=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | MaxAct=3.3624 | MeanAct=-0.0002 | GradNorm=1.7579 | Sparsity=0.0000 | HeadMeanAvg=0.8808 | Sink Val=0.0424 | Dead=0.000 | Open=0.000
Epoch 0 Step 500 | Loss=5.9388 | gate_mean=0.5522 | LayerMeans=[0.841, 0.442, 0.468, 0.501, 0.496, 0.561, 0.544, 0.565] | LayerStd=[0.113, 0.244, 0.265, 0.281, 0.289, 0.296, 0.297, 0.289] | LayerSparsity=[0.001, 0.064, 0.085, 0.082, 0.104, 0.074, 0.088, 0.068] | MaxAct=15.7813 | MeanAct=-3.8000 | GradNorm=0.5575 | Sparsity=0.0707 | HeadMeanAvg=0.5522 | Sink Val=0.0365 | Dead=0.001 | Open=0.002
Epoch 0 Step 1000 | Loss=5.5522 | gate_mean=0.3978 | LayerMeans=[0.817, 0.266, 0.283, 0.317, 0.304, 0.382, 0.379, 0.434] | LayerStd=[0.144, 0.242, 0.262,

In [ ]:
print(f'####Logs for Config 1 (With Gate) - SEED {SEED}')
print('train_loss: ', model_det['train_loss'])
print('val_loss_no_gate: ', model_det['val_loss'])
print('train_ppl_no_gate: ', model_det['train_ppl'])
print('val_ppl_no_gate: ', model_det['val_ppl'])

print('\n')

print('Best Training PPL: ', min(model_det['train_ppl']))
print('Best Validation PPL: ', min(model_det['val_ppl']))

####Logs for Config 1 (With Gate) - SEED 12
train_loss:  [4.695832725556691, 4.0634013029416405, 3.8667303202788035, 3.7520635724862417, 3.6729804850419363, 3.61296180173556, 3.5650904848416647, 3.525353162463506, 3.491419407304128, 3.4621813180446623]
val_loss_no_gate:  [4.227023003514608, 3.9931513382593793, 3.8884021489461262, 3.828101762072245, 3.79062712504069, 3.760871481513977, 3.7387651096343992, 3.7242687721888226, 3.715236792119344, 3.7007392265955605]
train_ppl_no_gate:  [109.4899457664804, 58.17183500269595, 47.78588582554666, 42.608917931699715, 39.36907009497943, 37.07570123248768, 35.34265123380086, 33.96576693225305, 32.83251736271542, 31.88645520968215]
val_ppl_no_gate:  [68.51296541674768, 54.22550329419145, 48.83279661678855, 45.97518351334228, 44.28416327733402, 42.9858710520326, 42.04603584807603, 41.44091891059397, 41.06831058542449, 40.477215137227994]


Best Training PPL:  31.88645520968215
Best Validation PPL:  40.477215137227994


In [ ]:
SEED2 = 42
model_det2 = model_training_with_gate(train_dataset, qk_norm = False, pre_norm = True, post_norm = False, vocab_size = vocab_size,
                                                                               n_transformer = 8, seed=SEED2)

train_dataloader size:  15000
val_dataloader size:  3750

--- Training with Gating ---
Epoch 0 Step 0 | Loss=11.0047 | gate_mean=0.8808 | LayerMeans=[0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881] | LayerStd=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | LayerSparsity=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | MaxAct=3.3297 | MeanAct=-0.0006 | GradNorm=1.5542 | Sparsity=0.0000 | HeadMeanAvg=0.8808 | Sink Val=0.0425 | Dead=0.000 | Open=0.000
Epoch 0 Step 500 | Loss=5.7960 | gate_mean=0.5269 | LayerMeans=[0.839, 0.473, 0.458, 0.456, 0.488, 0.466, 0.501, 0.533] | LayerStd=[0.113, 0.238, 0.271, 0.285, 0.292, 0.289, 0.294, 0.296] | LayerSparsity=[0.001, 0.047, 0.096, 0.117, 0.114, 0.121, 0.104, 0.087] | MaxAct=16.1716 | MeanAct=-3.8940 | GradNorm=0.5461 | Sparsity=0.0858 | HeadMeanAvg=0.5269 | Sink Val=0.0264 | Dead=0.002 | Open=0.001
Epoch 0 Step 1000 | Loss=5.7003 | gate_mean=0.3902 | LayerMeans=[0.816, 0.285, 0.293, 0.293, 0.323, 0.321, 0.377, 0.414] | LayerStd=[0.143, 0.244, 0.273,

In [ ]:
print('####Logs for Config 1 (With Gate) - SEED 42')
print('train_loss: ', model_det2['train_loss'])
print('val_loss: ', model_det2['val_loss'])
print('train_ppl: ', model_det2['train_ppl'])
print('val_ppl: ', model_det2['val_ppl'])

print('\n')

print('Best Training PPL: ', min(model_det2['train_ppl']))
print('Best Validation PPL: ', min(model_det2['val_ppl']))

####Logs for Config 1 (With Gate) - SEED 42
train_loss:  [4.695171550369262, 4.06536117088, 3.867521595923106, 3.751067053524653, 3.6705495802402495, 3.6099152299722035, 3.561807328335444, 3.5219020781517028, 3.488136146656672, 3.4587834485848745]
val_loss:  [4.22244681523641, 3.9931128671010336, 3.8874247756958007, 3.8269056887308754, 3.7887950277964273, 3.758713446935018, 3.7382276390075684, 3.7242097776412963, 3.712351743634542, 3.7046795351664223]
train_ppl:  [109.41757765770193, 58.28595591145939, 47.82371259686959, 42.56647848641602, 39.27348386103575, 36.96291933431236, 35.226806051446644, 33.84875024022869, 32.724896420980606, 31.77829306183935]
val_ppl:  [68.20015347656563, 54.22341721639501, 48.78509206401113, 45.92022669472196, 44.203104660137036, 42.893206078966315, 42.023443410762596, 41.43847419444694, 40.94999727008732, 40.637022493340865]


Best Training PPL:  31.77829306183935
Best Validation PPL:  40.637022493340865


In [ ]:
SEED3 = 100
model_det3 = model_training_with_gate(train_dataset, qk_norm = False, pre_norm = True, post_norm = False, vocab_size = vocab_size,
                                                                               n_transformer = 8, seed=SEED3)

train_dataloader size:  15000
val_dataloader size:  3750

--- Training with Gating ---
Epoch 0 Step 0 | Loss=11.0369 | gate_mean=0.8808 | LayerMeans=[0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881] | LayerStd=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | LayerSparsity=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | MaxAct=3.4242 | MeanAct=-0.0003 | GradNorm=1.5664 | Sparsity=0.0000 | HeadMeanAvg=0.8808 | Sink Val=0.0432 | Dead=0.000 | Open=0.000
Epoch 0 Step 500 | Loss=5.9724 | gate_mean=0.5524 | LayerMeans=[0.838, 0.456, 0.497, 0.526, 0.505, 0.511, 0.51, 0.576] | LayerStd=[0.118, 0.237, 0.27, 0.278, 0.291, 0.284, 0.286, 0.292] | LayerSparsity=[0.002, 0.054, 0.074, 0.069, 0.094, 0.08, 0.09, 0.07] | MaxAct=16.0727 | MeanAct=-3.5509 | GradNorm=0.5357 | Sparsity=0.0667 | HeadMeanAvg=0.5524 | Sink Val=0.0295 | Dead=0.001 | Open=0.002
Epoch 0 Step 1000 | Loss=5.4969 | gate_mean=0.4125 | LayerMeans=[0.812, 0.282, 0.344, 0.343, 0.338, 0.358, 0.374, 0.448] | LayerStd=[0.156, 0.24, 0.279, 0.285

In [ ]:
print('####Logs for Config 1 (With Gate) - SEED 100')
print('train_loss: ', model_det3['train_loss'])
print('val_loss: ', model_det3['val_loss'])
print('train_ppl: ', model_det3['train_ppl'])
print('val_ppl: ', model_det3['val_ppl'])

print('\n')

print('Best Training PPL: ', min(model_det3['train_ppl']))
print('Best Validation PPL: ', min(model_det3['val_ppl']))

####Logs for Config 1 (With Gate) - SEED 100
train_loss:  [4.695426764504115, 4.06630268090566, 3.8669270012378694, 3.750463472636541, 3.669701395003001, 3.6089420741558076, 3.5605549615065257, 3.520787629969915, 3.4869627485434216, 3.457577564096451]
val_loss:  [4.229072024154663, 4.000974505551656, 3.894057604598999, 3.834830281321208, 3.7942294573465984, 3.7626722693125405, 3.7453865177790324, 3.729730738258362, 3.7172785110473634, 3.7076792475382487]
train_ppl:  [109.44550613384081, 58.340858564948896, 47.7952853237215, 42.540793925662136, 39.240186794845705, 36.926966151233216, 35.18271678179104, 33.81104857424633, 32.686519609306416, 31.73999520720285]
val_ppl:  [68.65349382063076, 54.65138216675384, 49.109750745177244, 46.28557147526005, 44.44397722800405, 43.06334922406973, 42.325363563146404, 41.66788708554649, 41.15224618978668, 40.75910488695736]


Best Training PPL:  31.73999520720285
Best Validation PPL:  40.75910488695736


### config 2: pre norm =True, post_norm = False, qk norm = True

In [ ]:
model_det = model_training_with_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = False, vocab_size = vocab_size,
                                                                               n_transformer = 8, seed=SEED)

train_dataloader size:  15000
val_dataloader size:  3750

--- Training with Gating ---
Epoch 0 Step 0 | Loss=10.9605 | gate_mean=0.8808 | LayerMeans=[0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881] | LayerStd=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | LayerSparsity=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | MaxAct=3.3866 | MeanAct=-0.0002 | GradNorm=1.7546 | Sparsity=0.0000 | HeadMeanAvg=0.8808 | Sink Val=0.0422 | Dead=0.000 | Open=0.000
Epoch 0 Step 500 | Loss=5.8988 | gate_mean=0.5842 | LayerMeans=[0.84, 0.497, 0.521, 0.527, 0.536, 0.576, 0.584, 0.591] | LayerStd=[0.115, 0.247, 0.27, 0.285, 0.288, 0.295, 0.303, 0.292] | LayerSparsity=[0.001, 0.041, 0.059, 0.078, 0.081, 0.074, 0.081, 0.065] | MaxAct=15.7915 | MeanAct=-3.8140 | GradNorm=0.5501 | Sparsity=0.0599 | HeadMeanAvg=0.5842 | Sink Val=0.0319 | Dead=0.001 | Open=0.003
Epoch 0 Step 1000 | Loss=5.5493 | gate_mean=0.4285 | LayerMeans=[0.812, 0.313, 0.334, 0.353, 0.346, 0.403, 0.419, 0.447] | LayerStd=[0.151, 0.267, 0.283, 0

In [ ]:
print('#### Logs for Config 2 (With Gate) - SEED 12')
print('train_loss: ', model_det['train_loss'])
print('val_loss: ', model_det['val_loss'])
print('train_ppl: ', model_det['train_ppl'])
print('val_ppl: ', model_det['val_ppl'])
print('\nBest Training PPL: ', min(model_det['train_ppl']))
print('Best Validation PPL: ', min(model_det['val_ppl']))

#### Logs for Config 2 (With Gate) - SEED 12
train_loss:  [4.677497733640671, 4.054813833030065, 3.8610034872214, 3.7481135477224985, 3.669686558834712, 3.610477603260676, 3.5632950085163118, 3.52421307776769, 3.490927352841695, 3.462142832740148]
val_loss:  [4.213144381904602, 3.988619053586324, 3.884658376757304, 3.822900394821167, 3.783943663978577, 3.758944452857971, 3.740096597099304, 3.725758367093404, 3.7143018840789797, 3.702404843711853]
train_ppl:  [107.5007402553912, 57.67442492372607, 47.51300614889385, 42.44094362046201, 39.23960462514932, 36.98371213892089, 35.279251273831, 33.927065147040466, 32.81636595004742, 31.885228073357034]
val_ppl:  [67.56866782602124, 53.980293975689705, 48.650319540344746, 45.73667053398524, 43.989178653571415, 42.903115808184005, 42.10205690514786, 41.502695091563666, 41.02993343400303, 40.544690858338704]

Best Training PPL:  31.885228073357034
Best Validation PPL:  40.544690858338704


In [ ]:
SEED2 = 42
model_det2 = model_training_with_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = False, vocab_size = vocab_size,
                                                                               n_transformer = 8, seed=SEED2)

train_dataloader size:  15000
val_dataloader size:  3750

--- Training with Gating ---
Epoch 0 Step 0 | Loss=11.0048 | gate_mean=0.8808 | LayerMeans=[0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881] | LayerStd=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | LayerSparsity=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | MaxAct=3.4270 | MeanAct=-0.0005 | GradNorm=1.5469 | Sparsity=0.0000 | HeadMeanAvg=0.8808 | Sink Val=0.0429 | Dead=0.000 | Open=0.000
Epoch 0 Step 500 | Loss=5.7823 | gate_mean=0.5538 | LayerMeans=[0.838, 0.504, 0.519, 0.486, 0.519, 0.504, 0.522, 0.538] | LayerStd=[0.116, 0.243, 0.273, 0.294, 0.304, 0.297, 0.294, 0.307] | LayerSparsity=[0.001, 0.04, 0.064, 0.117, 0.113, 0.114, 0.094, 0.105] | MaxAct=15.8534 | MeanAct=-3.8639 | GradNorm=0.5471 | Sparsity=0.0811 | HeadMeanAvg=0.5538 | Sink Val=0.0281 | Dead=0.003 | Open=0.002
Epoch 0 Step 1000 | Loss=5.6645 | gate_mean=0.4195 | LayerMeans=[0.812, 0.29, 0.353, 0.321, 0.38, 0.356, 0.403, 0.44] | LayerStd=[0.149, 0.258, 0.293, 0.3

In [ ]:
print('#### Logs for Config 2 (With Gate) - SEED 42')
print('train_loss: ', model_det2['train_loss'])
print('val_loss: ', model_det2['val_loss'])
print('train_ppl: ', model_det2['train_ppl'])
print('val_ppl: ', model_det2['val_ppl'])
print('\nBest Training PPL: ', min(model_det2['train_ppl']))
print('Best Validation PPL: ', min(model_det2['val_ppl']))

#### Logs for Config 2 (With Gate) - SEED 42
train_loss:  [4.674583984629313, 4.048083135430018, 3.8540851747035982, 3.7411240969022117, 3.662707894484202, 3.6038171689828236, 3.556957592169444, 3.5177689365228018, 3.4846543903191884, 3.455996220334371]
val_loss:  [4.202537799708049, 3.9783079634984335, 3.8743236846923828, 3.816750860595703, 3.777504063288371, 3.7509503920237224, 3.730872834587097, 3.7187025641759237, 3.7045137314478556, 3.6991321554819745]
train_ppl:  [107.18796597391461, 57.28753927602419, 47.18543076469807, 42.145338992680166, 38.96671789517514, 36.73820306100053, 35.05637893344138, 33.70913728034174, 32.61115443132431, 31.689843028142324]
val_ppl:  [66.85578251838166, 53.42655801823298, 48.15012261163926, 45.45627434978275, 43.70681603536899, 42.56151290679807, 41.71550301222477, 41.2108909214167, 40.630285282442735, 40.41221761583106]

Best Training PPL:  31.689843028142324
Best Validation PPL:  40.41221761583106


In [ ]:
SEED3 = 100
model_det3 = model_training_with_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = False, vocab_size = vocab_size,
                                                                               n_transformer = 8, seed=SEED3)

train_dataloader size:  15000
val_dataloader size:  3750

--- Training with Gating ---
Epoch 0 Step 0 | Loss=11.0359 | gate_mean=0.8808 | LayerMeans=[0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881] | LayerStd=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | LayerSparsity=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | MaxAct=3.4045 | MeanAct=-0.0004 | GradNorm=1.5553 | Sparsity=0.0000 | HeadMeanAvg=0.8808 | Sink Val=0.0434 | Dead=0.000 | Open=0.000
Epoch 0 Step 500 | Loss=5.9413 | gate_mean=0.5818 | LayerMeans=[0.838, 0.498, 0.541, 0.588, 0.525, 0.549, 0.547, 0.568] | LayerStd=[0.115, 0.241, 0.279, 0.293, 0.292, 0.289, 0.295, 0.29] | LayerSparsity=[0.001, 0.037, 0.066, 0.068, 0.091, 0.072, 0.087, 0.07] | MaxAct=16.0255 | MeanAct=-3.6054 | GradNorm=0.5208 | Sparsity=0.0615 | HeadMeanAvg=0.5818 | Sink Val=0.0331 | Dead=0.001 | Open=0.003
Epoch 0 Step 1000 | Loss=5.4710 | gate_mean=0.4317 | LayerMeans=[0.81, 0.302, 0.368, 0.405, 0.358, 0.375, 0.39, 0.445] | LayerStd=[0.154, 0.261, 0.296, 0.3

In [ ]:
print('#### Logs for Config 2 (With Gate) - SEED 100')
print('train_loss: ', model_det3['train_loss'])
print('val_loss: ', model_det3['val_loss'])
print('train_ppl: ', model_det3['train_ppl'])
print('val_ppl: ', model_det3['val_ppl'])
print('\nBest Training PPL: ', min(model_det3['train_ppl']))
print('Best Validation PPL: ', min(model_det3['val_ppl']))

#### Logs for Config 2 (With Gate) - SEED 100
train_loss:  [4.672028570572535, 4.0468161436875665, 3.8508571123600004, 3.7370175007184345, 3.6582455904483795, 3.598961913061142, 3.5517559236049654, 3.51274056353569, 3.4794756201903025, 3.4504711392084757]
val_loss:  [4.2063358335495, 3.9837602863311767, 3.8788534534454344, 3.8217034233729046, 3.780849390411377, 3.75204243850708, 3.73317264251709, 3.7206431910832722, 3.708281772295634, 3.701049018796285]
train_ppl:  [106.91440601729207, 57.21500239833203, 47.03335883337162, 41.97261999073556, 38.7932239319414, 36.560262007271874, 34.874500714361346, 33.540060611551404, 32.44270531499167, 31.515236875301028]
val_ppl:  [67.11018585330565, 53.71865243330438, 48.368726270722824, 45.68195779654219, 43.85327447161464, 42.6080174452315, 41.81155106055343, 41.290943536220354, 40.783670656739666, 40.48975660530874]

Best Training PPL:  31.515236875301028
Best Validation PPL:  40.48975660530874


### config 3: post_norm =True, post_norm = True, qk norm = True

In [ ]:
model_det = model_training_with_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = True, vocab_size = vocab_size,
                                                                               n_transformer = 8, seed=SEED)

train_dataloader size:  15000
val_dataloader size:  3750

--- Training with Gating ---
Epoch 0 Step 0 | Loss=10.9528 | gate_mean=0.8808 | LayerMeans=[0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881] | LayerStd=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | LayerSparsity=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | MaxAct=3.2551 | MeanAct=0.0005 | GradNorm=3.6987 | Sparsity=0.0000 | HeadMeanAvg=0.8808 | Sink Val=0.0443 | Dead=0.000 | Open=0.000
Epoch 0 Step 500 | Loss=6.0016 | gate_mean=0.7299 | LayerMeans=[0.839, 0.652, 0.696, 0.717, 0.748, 0.69, 0.75, 0.747] | LayerStd=[0.128, 0.267, 0.275, 0.266, 0.242, 0.286, 0.241, 0.253] | LayerSparsity=[0.001, 0.029, 0.037, 0.032, 0.016, 0.045, 0.015, 0.021] | MaxAct=15.8380 | MeanAct=-3.7533 | GradNorm=0.6440 | Sparsity=0.0244 | HeadMeanAvg=0.7299 | Sink Val=0.0343 | Dead=0.000 | Open=0.014
Epoch 0 Step 1000 | Loss=5.6107 | gate_mean=0.6136 | LayerMeans=[0.82, 0.504, 0.563, 0.572, 0.621, 0.559, 0.617, 0.651] | LayerStd=[0.16, 0.316, 0.33, 0.336

In [ ]:
print(f'#### Logs for Config 3 (With Gate) - SEED {SEED}')
print('train_loss: ', model_det['train_loss'])
print('val_loss_no_gate: ', model_det['val_loss'])
print('train_ppl_no_gate: ', model_det['train_ppl'])
print('val_ppl_no_gate: ', model_det['val_ppl'])

print('\n')

print('Best Training PPL: ', min(model_det['train_ppl']))
print('Best Validation PPL: ', min(model_det['val_ppl']))

#### Logs for Config 3 (With Gate) - SEED 12
train_loss:  [4.693947413667043, 4.039166353750229, 3.839146086661021, 3.7214957962989805, 3.639353790807724, 3.576455140765508, 3.5259627953211465, 3.483648306798935, 3.447443167289098, 3.4158053368409473]
val_loss_no_gate:  [4.205860771814982, 3.97514150651296, 3.8727877668380737, 3.80864072303772, 3.769423676363627, 3.7443810899098713, 3.7228513161977133, 3.7109914608637493, 3.7012292214075724, 3.689115228907267]
train_ppl_no_gate:  [109.28371753328477, 56.778989478605126, 46.48576267417898, 41.32616342223656, 38.06722938195846, 35.74659932236434, 33.98647989281907, 32.5783613853122, 31.41995395281793, 30.441455183898167]
val_ppl_no_gate:  [67.07831194364724, 53.257652677261454, 48.076224743687625, 45.089108606778325, 43.35507108207666, 42.28282984892153, 41.38221984330746, 40.89433156347114, 40.49705362263046, 40.00943210361568]


Best Training PPL:  30.441455183898167
Best Validation PPL:  40.00943210361568


In [ ]:
SEED2 = 42
model_det2 = model_training_with_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = True, vocab_size = vocab_size,
                                                                               n_transformer = 8, seed=SEED2)

train_dataloader size:  15000
val_dataloader size:  3750

--- Training with Gating ---
Epoch 0 Step 0 | Loss=10.9753 | gate_mean=0.8808 | LayerMeans=[0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881] | LayerStd=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | LayerSparsity=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | MaxAct=3.5888 | MeanAct=-0.0024 | GradNorm=3.3174 | Sparsity=0.0000 | HeadMeanAvg=0.8808 | Sink Val=0.0467 | Dead=0.000 | Open=0.000
Epoch 0 Step 500 | Loss=5.8460 | gate_mean=0.7238 | LayerMeans=[0.838, 0.662, 0.667, 0.758, 0.721, 0.707, 0.748, 0.688] | LayerStd=[0.131, 0.267, 0.294, 0.237, 0.266, 0.274, 0.248, 0.289] | LayerSparsity=[0.003, 0.032, 0.06, 0.017, 0.034, 0.033, 0.021, 0.046] | MaxAct=16.1630 | MeanAct=-3.9300 | GradNorm=0.6038 | Sparsity=0.0307 | HeadMeanAvg=0.7238 | Sink Val=0.0293 | Dead=0.001 | Open=0.016
Epoch 0 Step 1000 | Loss=5.7720 | gate_mean=0.6196 | LayerMeans=[0.822, 0.527, 0.554, 0.679, 0.615, 0.57, 0.632, 0.558] | LayerStd=[0.157, 0.318, 0.34, 0.

In [ ]:
print('#### Logs for Config 3 (With Gate) - SEED 42')
print('train_loss: ', model_det2['train_loss'])
print('val_loss: ', model_det2['val_loss'])
print('train_ppl: ', model_det2['train_ppl'])
print('val_ppl: ', model_det2['val_ppl'])
print('\nBest Training PPL: ', min(model_det2['train_ppl']))
print('Best Validation PPL: ', min(model_det2['val_ppl']))

#### Logs for Config 3 (With Gate) - SEED 42
train_loss:  [4.6967745681285855, 4.041444413312276, 3.8421232211589813, 3.725096893453598, 3.64279905722936, 3.57989799434344, 3.52950784184138, 3.486871440633138, 3.4505379260381064, 3.4185952102820076]
val_loss:  [4.2014864126841225, 3.974108350245158, 3.872968837865194, 3.8136519715627033, 3.775611585553487, 3.746514253679911, 3.7268699841181436, 3.713372045389811, 3.70188271522522, 3.697400878461202]
train_ppl:  [109.59311663632973, 56.90848283930345, 46.624363255814906, 41.4752512304757, 38.1986073151506, 35.86988172960403, 34.10717735775817, 32.68353520775825, 31.517341748345107, 30.526501570387964]
val_ppl:  [66.78552815490215, 53.20265761372868, 48.08493074326038, 45.31562843531333, 43.62418007717807, 42.373122319724914, 41.54885584578678, 40.99179994622575, 40.52352684590753, 40.34231340135484]

Best Training PPL:  30.526501570387964
Best Validation PPL:  40.34231340135484


In [ ]:
SEED3 = 100
model_det3 = model_training_with_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = True, vocab_size = vocab_size,
                                                                               n_transformer = 8, seed=SEED3)

train_dataloader size:  15000
val_dataloader size:  3750

--- Training with Gating ---
Epoch 0 Step 0 | Loss=11.0450 | gate_mean=0.8808 | LayerMeans=[0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881] | LayerStd=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | LayerSparsity=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | MaxAct=3.1297 | MeanAct=0.0008 | GradNorm=3.2308 | Sparsity=0.0000 | HeadMeanAvg=0.8808 | Sink Val=0.0477 | Dead=0.000 | Open=0.000
Epoch 0 Step 500 | Loss=6.0128 | gate_mean=0.7234 | LayerMeans=[0.837, 0.693, 0.665, 0.692, 0.713, 0.719, 0.682, 0.784] | LayerStd=[0.132, 0.246, 0.29, 0.281, 0.267, 0.271, 0.283, 0.208] | LayerSparsity=[0.002, 0.016, 0.056, 0.038, 0.028, 0.036, 0.044, 0.005] | MaxAct=15.9879 | MeanAct=-3.6261 | GradNorm=0.5878 | Sparsity=0.0282 | HeadMeanAvg=0.7234 | Sink Val=0.0388 | Dead=0.001 | Open=0.015
Epoch 0 Step 1000 | Loss=5.5548 | gate_mean=0.6150 | LayerMeans=[0.819, 0.578, 0.524, 0.566, 0.606, 0.595, 0.552, 0.681] | LayerStd=[0.162, 0.303, 0.339, 0

In [ ]:
print('#### Logs for Config 3 (With Gate) - SEED 100')
print('train_loss: ', model_det3['train_loss'])
print('val_loss: ', model_det3['val_loss'])
print('train_ppl: ', model_det3['train_ppl'])
print('val_ppl: ', model_det3['val_ppl'])
print('\nBest Training PPL: ', min(model_det3['train_ppl']))
print('Best Validation PPL: ', min(model_det3['val_ppl']))

#### Logs for Config 3 (With Gate) - SEED 100
train_loss:  [4.689779433965683, 4.035689094861349, 3.8366784206231435, 3.7197213768641153, 3.637693693971634, 3.5750467362562817, 3.5246779846350353, 3.4825407517910003, 3.446429480902354, 3.414893560552597]
val_loss:  [4.202786520512899, 3.977637047068278, 3.8737398244222003, 3.814835604476929, 3.7760665137608846, 3.7469120774586995, 3.7292746772766114, 3.7158823624928794, 3.705293117904663, 3.6971534457524617]
train_ppl:  [108.82917314084797, 56.58189710194071, 46.371192754671, 41.25289849527016, 38.00408652103597, 35.69628908758628, 33.94284173965516, 32.54229903221589, 31.388120110715963, 30.413712036572548]
val_ppl:  [66.8724130105075, 53.39072528438093, 48.12201787346032, 45.36929726048638, 43.64403046212743, 42.389982708874705, 41.64888832068176, 41.09483162970876, 40.661964320030926, 40.332332628308876]

Best Training PPL:  30.413712036572548
Best Validation PPL:  40.332332628308876


In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')
